# BagZITboost — combined val-best (HP #11 + PP #4) + position + die_x/die_y

**목적**: combined_best 노트북의 변형. **xy 추가** 효과를 확인.

**Combined params** (combined_best 와 동일):
- **Model HP** = `bag-zit-hpo-001` trial **#11**
- **PP params** = `bag-zit-pp-hpo-001` trial **#4**
- `tau_pi` = 0.8689 (avg)

**학습 입력 차이 (vs combined_best)**:
- combined_best: X feature + `position` (1~4)
- **이 노트북**: X feature + `position` + **`die_x` + `die_y`** (run_wf_xy 파싱)

die_x/die_y = wafer 내 절대 좌표 (예: x=12~66, y=11~32). position = unit 내 슬롯(1~4).

**격리**: 출력 `4_output/_temp/bag_zit_combined_best_xy/`. combined_best 산출물 보존.

## 1. 환경 + import

In [1]:
import os, sys

%run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL,
    SPLIT_COL, OUTPUT_DIR,
)
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

PP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_ROOT not in sys.path:
    sys.path.insert(0, PP_ROOT)

from final.modules import preprocess
from modules.zi_tweedie import ZITboostRegressor
from meta_features import parse_run_wf_xy   # ★ xy 노트북 추가

import lightgbm as lgb
from sklearn.model_selection import KFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. 실험 설정 (combined val-best params, position + xy 포함)

In [2]:
# ── 실험 식별 ──
EXP_ID = 'bag-zit-combined-best-xy-001'
USER   = 'jh'
N_FOLDS = 5
CLIP_Y_EXTREME = True

# ── 학습 입력 추가 토글 ──
USE_POSITION   = True   # position(1~4)
USE_DIE_COORDS = True   # ★ die_x, die_y 추가

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_combined_best_xy')
os.makedirs(OUT_DIR, exist_ok=True)

# ── PP params: bag-zit-pp-hpo-001 trial #4 ──
PARAMS = {
    'missing_threshold':          0.38347071112677134,
    'corr_threshold':             0.9021910061199376,
    'corr_keep_by':               'std',
    'add_indicator':              False,
    'indicator_threshold':        0.07278178485435405,
    'spatial_max_dist':           2.0,
    'post_impute_corr_threshold': 0.9542261548497027,
    'post_impute_corr_keep_by':   'std',
}

# ── Model HP: bag-zit-hpo-001 trial #11 ──
FIXED_HP = dict(
    zeta                  = 1.3723978853416428,
    n_em_iters            = 14,
    em_tol                = 1e-7,
    mu_n_estimators       = 221,
    mu_learning_rate      = 0.01816196092838566,
    mu_num_leaves         = 175,
    mu_max_depth          = 3,
    mu_min_child_samples  = 34,
    mu_subsample          = 0.7106709866201101,
    mu_colsample_bytree   = 0.6857939132165285,
    mu_reg_alpha          = 7.289745272076977e-05,
    mu_reg_lambda         = 0.013426470278290088,
    pi_n_estimators       = 172,
    pi_learning_rate      = 0.04263245149297556,
    pi_num_leaves         = 168,
    pi_max_depth          = 11,
    pi_min_child_samples  = 47,
    phi_n_estimators      = 179,
    phi_learning_rate     = 0.005470341479056198,
    phi_num_leaves        = 104,
    phi_max_depth         = 5,
    phi_min_child_samples = 167,
    random_state          = SEED,
    n_jobs                = -1,
    verbose               = -1,
    device                = 'cpu',
)

# ── τ_π (두 trial 평균) ──
TAU_PI_FROM_HPO    = 0.8663904720840038
TAU_PI_FROM_PP_HPO = 0.8714562157320731
TAU_PI = (TAU_PI_FROM_HPO + TAU_PI_FROM_PP_HPO) / 2.0

print(f'EXP_ID={EXP_ID} | N_FOLDS={N_FOLDS}')
print(f'OUT_DIR={OUT_DIR}')
print(f'USE_POSITION={USE_POSITION}, USE_DIE_COORDS={USE_DIE_COORDS}')
print(f'PARAMS keys: {list(PARAMS)}')
print(f'FIXED_HP keys: {len(FIXED_HP)} (model HP from hpo trial #11)')
print(f'TAU_PI: {TAU_PI:.4f}')

EXP_ID=bag-zit-combined-best-xy-001 | N_FOLDS=5
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_combined_best_xy
USE_POSITION=True, USE_DIE_COORDS=True
PARAMS keys: ['missing_threshold', 'corr_threshold', 'corr_keep_by', 'add_indicator', 'indicator_threshold', 'spatial_max_dist', 'post_impute_corr_threshold', 'post_impute_corr_keep_by']
FIXED_HP keys: 26 (model HP from hpo trial #11)
TAU_PI: 0.8689


## 3. 데이터 로드 + run_wf_xy 파싱 (die_x, die_y 추가) + Y clip

In [3]:
xs, ys = load_all()

# ── run_wf_xy 파싱 → lot, wafer_no, die_x, die_y 추가 ──
if USE_DIE_COORDS:
    xs = parse_run_wf_xy(xs, inplace=False, verbose=True)
    print(f'  parse_run_wf_xy 후 xs.shape={xs.shape}')
    print(f'  die_x range={xs["die_x"].min()}~{xs["die_x"].max()}, '
          f'die_y range={xs["die_y"].min()}~{xs["die_y"].max()}')

feat_cols = get_feat_cols(xs)   # X0~X1086 (die_x/die_y는 'X' 시작 안 함 → 미포함)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'\n[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

print(f'\n[데이터 로드 완료]')
print(f'  xs: {xs.shape}, X feat_cols: {len(feat_cols)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')
print(f'  position 분포: {dict(xs["position"].value_counts().sort_index())}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[run_wf_xy 파싱] lot: 28개, wafer: 25개, die_x: 12~66, die_y: 11~32
  parse_run_wf_xy 후 xs.shape=(174572, 1095)
  die_x range=12~66, die_y range=11~32

[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개 샘플

[데이터 로드 완료]
  xs: (174572, 1095), X feat_cols: 1087
  unit train=26,187, val=8,727, test=8,729
  position 분포: {1: np.int64(43643), 2: np.int64(43643), 3: np.int64(43643), 4: np.int64(43643)}


## 4. 전처리 (PP params trial #4 적용) + position/die_x/die_y 학습 입력에 추가

**주의**: `preprocess.run`은 `feat_cols`(=X*)만 cleaning. position, die_x, die_y는 cleaning이 건드리지 않으므로 xs_train_die에 그대로 살아있음. cleaning 후 `feat_cols_clean`에 append.

In [4]:
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

print(f'\n[cleaning 완료] feat_cols_clean (X+indicator): {len(feat_cols_clean)}')

# ── position 추가 ──
if USE_POSITION:
    for split_name, df in [('train', xs_train_die), ('val', xs_val_die), ('test', xs_test_die)]:
        assert 'position' in df.columns, f'{split_name} 에 position 없음'
        assert df['position'].isin([1, 2, 3, 4]).all(), f'{split_name} position 값 이상'
    feat_cols_clean = list(feat_cols_clean) + ['position']
    print(f'[position 추가] feat_cols_clean: {len(feat_cols_clean)-1} → {len(feat_cols_clean)}')

# ── die_x, die_y 추가 ──
if USE_DIE_COORDS:
    # cleaning 후 xs_train_die에 die_x/die_y가 살아있는지 확인
    for split_name, df in [('train', xs_train_die), ('val', xs_val_die), ('test', xs_test_die)]:
        for col in ['die_x', 'die_y']:
            if col not in df.columns:
                # cleaning 단계가 비-X 컬럼 일부를 떨어뜨렸을 가능성 → reindex 부착
                df[col] = xs[col].reindex(df.index).values
                print(f'  [die-coord reattach] {split_name}/{col} ← reindex from xs')
        assert df['die_x'].notna().all(), f'{split_name} die_x NaN 존재'
        assert df['die_y'].notna().all(), f'{split_name} die_y NaN 존재'
    feat_cols_clean = list(feat_cols_clean) + ['die_x', 'die_y']
    print(f'[die_x/die_y 추가] feat_cols_clean: {len(feat_cols_clean)-2} → {len(feat_cols_clean)}')

# ── numpy 변환 ──
X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y 존재'

n_train_die, n_val_die, n_test_die = len(X_train_die), len(X_val_die), len(X_test_die)
print(f'\n[X 구성 완료]')
print(f'  X_train_die: {X_train_die.shape}')
print(f'  X_val_die:   {X_val_die.shape}')
print(f'  X_test_die:  {X_test_die.shape}')
print(f'  feat_cols_clean[-5:] = {feat_cols_clean[-5:]}  ← 마지막에 position, die_x, die_y 포함 확인')

[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 990)

[고결측 제거] threshold=38%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 985)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 958)

[고상관 제거] threshold=0.9021910061199376, keep_by=std (std)
  제거: 328개, 잔여: 568개
    컬럼: 896 → 568 (328개 제거)
    DataFrame: (104748, 630)

[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=2.0): 75,373개 채움 → 잔여: 268,121
  2단계 (lot 평균, train 기준): 186,925개 채움 → 잔여: 81,196
  3단계 (train 전체 평균): 81,196개 채움 → 잔여: 0

  [요약] 343,494 → 공간(75,373) → lot(186,925) → 전체(81,196) → 잔여(0)

[고상관 제거] threshold=0.9542261548497027, keep_by=std (std)
  제거: 0개, 잔여: 568개
    [고상관 제거 2차 / imputation 후] threshold=0.9542261548497027
    컬럼: 568 → 568 (0개 제거)
    DataFrame: (104748, 630)

클

## 5. BagZITboostRegressor 정의 (combined_best 노트북과 동일)

In [5]:
class BagZITboostRegressor(ZITboostRegressor):
    """ZITboost + bag (unit) constraint via B3 allocation."""

    @staticmethod
    def _allocate_b3(unit_y_per_unit, contribution, inverse, n_units):
        contrib_sum_per_unit = np.zeros(n_units)
        np.add.at(contrib_sum_per_unit, inverse, contribution)
        contrib_sum_die = contrib_sum_per_unit[inverse]
        n_die_per_unit = np.bincount(inverse, minlength=n_units).astype(np.float64)
        n_die_die = n_die_per_unit[inverse]
        share = np.where(
            contrib_sum_die > 1e-12,
            contribution / np.maximum(contrib_sum_die, 1e-12),
            1.0 / np.maximum(n_die_die, 1.0),
        )
        return unit_y_per_unit[inverse] * share

    def fit(self, X, y, unit_id):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        unit_id = np.asarray(unit_id)

        unique_units, first_idx, inverse = np.unique(
            unit_id, return_index=True, return_inverse=True
        )
        n_units = len(unique_units)
        unit_y_per_unit = y[first_idx]

        n_die_per_unit = np.bincount(inverse, minlength=n_units).astype(np.float64)
        y_die_alloc = unit_y_per_unit[inverse] / np.maximum(n_die_per_unit[inverse], 1.0)

        pi_arr, mu_arr, phi_arr = self._initialize(X, y_die_alloc)
        self._phi_current = phi_arr

        self.em_history_ = []
        prev_rmse = np.inf
        em_iter = 0

        for em_iter in range(self.n_em_iters):
            if em_iter > 0:
                contribution = np.clip((1 - pi_arr) * mu_arr, 0, None)
                y_die_alloc = self._allocate_b3(
                    unit_y_per_unit, contribution, inverse, n_units
                )

            posterior = self._e_step(y_die_alloc, pi_arr, mu_arr, phi_arr)

            self._phi_current = phi_arr
            lgb_pi, lgb_mu, lgb_phi, pi_arr, mu_arr, phi_arr = \
                self._m_step(X, y_die_alloc, posterior)

            pred_die = np.clip((1 - pi_arr) * mu_arr, 0, None)
            pred_unit = np.zeros(n_units)
            np.add.at(pred_unit, inverse, pred_die)
            rmse_unit = float(np.sqrt(np.mean((unit_y_per_unit - pred_unit) ** 2)))

            self.em_history_.append({
                'iter':      em_iter + 1,
                'unit_rmse': rmse_unit,
                'pi_mean':   float(pi_arr.mean()),
                'mu_mean':   float(mu_arr.mean()),
            })

            rmse_delta = prev_rmse - rmse_unit
            if em_iter >= 2 and abs(rmse_delta) < self.em_tol:
                break
            prev_rmse = rmse_unit

        self.n_em_iters_actual_ = em_iter + 1
        self.lgb_pi_ = lgb_pi
        self.lgb_mu_ = lgb_mu
        self.lgb_phi_ = lgb_phi
        self.fitted_ = True
        return self


print('BagZITboostRegressor 정의 완료')

BagZITboostRegressor 정의 완료


## 6. KFold split (combined_best 노트북과 동일)

In [6]:
unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))

print(f'fold split: {N_FOLDS} folds, {len(FOLDS)} 개')
print(f'  unit_ids_train_unique: {len(unit_ids_train_unique):,}')
for fi, (tr_idx, vl_idx) in enumerate(FOLDS):
    print(f'  fold {fi+1}: train={len(tr_idx):,} val={len(vl_idx):,}')

fold split: 5 folds, 5 개
  unit_ids_train_unique: 26,187
  fold 1: train=20,949 val=5,238
  fold 2: train=20,949 val=5,238
  fold 3: train=20,950 val=5,237
  fold 4: train=20,950 val=5,237
  fold 5: train=20,950 val=5,237


## 7. 5-fold refit + die-level 캡쳐

In [7]:
import time

def _sum_die_to_unit(pred_die, uid_die):
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    n_units = len(unique_units)
    pred_unit = np.zeros(n_units)
    np.add.at(pred_unit, inverse, pred_die)
    return pred_unit, unique_units

oof_die_pi   = np.full(n_train_die, np.nan)
oof_die_mu   = np.full(n_train_die, np.nan)
oof_die_pred = np.full(n_train_die, np.nan)

val_die_pi   = np.zeros(n_val_die)
val_die_mu   = np.zeros(n_val_die)
val_die_pred = np.zeros(n_val_die)

test_die_pi   = np.zeros(n_test_die)
test_die_mu   = np.zeros(n_test_die)
test_die_pred = np.zeros(n_test_die)

print(f'=== 5-fold refit (combined val-best HP+PP, position+xy) ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unit_ids_train_unique[tr_uidx]
    vl_units = unit_ids_train_unique[vl_uidx]
    tr_die_mask = np.isin(uid_train_die, tr_units)
    vl_die_mask = np.isin(uid_train_die, vl_units)

    X_tr   = X_train_die[tr_die_mask]
    X_vl   = X_train_die[vl_die_mask]
    y_tr   = y_train_die_broadcast[tr_die_mask]
    uid_tr = uid_train_die[tr_die_mask]

    model = BagZITboostRegressor(**FIXED_HP)
    model.fit(X_tr, y_tr, unit_id=uid_tr)

    pi_vl, mu_vl, _ = model.predict_components(X_vl)
    pred_vl = np.clip((1 - pi_vl) * mu_vl, 0, None)
    oof_die_pi[vl_die_mask]   = pi_vl
    oof_die_mu[vl_die_mask]   = mu_vl
    oof_die_pred[vl_die_mask] = pred_vl

    pi_v, mu_v, _ = model.predict_components(X_val_die)
    pi_t, mu_t, _ = model.predict_components(X_test_die)
    val_die_pi   += pi_v / N_FOLDS
    val_die_mu   += mu_v / N_FOLDS
    val_die_pred += np.clip((1 - pi_v) * mu_v, 0, None) / N_FOLDS
    test_die_pi   += pi_t / N_FOLDS
    test_die_mu   += mu_t / N_FOLDS
    test_die_pred += np.clip((1 - pi_t) * mu_t, 0, None) / N_FOLDS

    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_pi).any(),   'OOF die π 미커버'
assert not np.isnan(oof_die_mu).any(),   'OOF die μ 미커버'
assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

print(f'\n[refit 완료] die-level 캡쳐 OK ({time.time()-t0:.0f}s)')

=== 5-fold refit (combined val-best HP+PP, position+xy) ===
  fold 1/5 done (503s)
  fold 2/5 done (938s)
  fold 3/5 done (1423s)
  fold 4/5 done (1942s)
  fold 5/5 done (2337s)

[refit 완료] die-level 캡쳐 OK (2337s)


## 8. τ_π 적용 + raw vs clipped 비교

In [8]:
oof_die_pred_clipped  = np.where(oof_die_pi  > TAU_PI, 0.0, oof_die_pred)
val_die_pred_clipped  = np.where(val_die_pi  > TAU_PI, 0.0, val_die_pred)
test_die_pred_clipped = np.where(test_die_pi > TAU_PI, 0.0, test_die_pred)

oof_unit_raw_arr,  oof_unit_ids  = _sum_die_to_unit(oof_die_pred,         uid_train_die)
oof_unit_clip_arr, _              = _sum_die_to_unit(oof_die_pred_clipped, uid_train_die)
val_unit_raw_arr,  val_unit_ids  = _sum_die_to_unit(val_die_pred,         uid_val_die)
val_unit_clip_arr, _              = _sum_die_to_unit(val_die_pred_clipped, uid_val_die)
test_unit_raw_arr, test_unit_ids = _sum_die_to_unit(test_die_pred,        uid_test_die)
test_unit_clip_arr, _             = _sum_die_to_unit(test_die_pred_clipped, uid_test_die)

oof_unit_raw   = pd.Series(oof_unit_raw_arr,  index=oof_unit_ids).reindex(y_train_unit.index)
oof_unit_clip  = pd.Series(oof_unit_clip_arr, index=oof_unit_ids).reindex(y_train_unit.index)
val_unit_raw   = pd.Series(val_unit_raw_arr,  index=val_unit_ids).reindex(y_val_unit.index)
val_unit_clip  = pd.Series(val_unit_clip_arr, index=val_unit_ids).reindex(y_val_unit.index)
test_unit_raw  = pd.Series(test_unit_raw_arr, index=test_unit_ids).reindex(y_test_unit.index)
test_unit_clip = pd.Series(test_unit_clip_arr, index=test_unit_ids).reindex(y_test_unit.index)

def _rmse(pred, true):
    return float(np.sqrt(np.mean((pred.values - true.values) ** 2)))

oof_rmse_raw      = _rmse(oof_unit_raw,   y_train_unit)
oof_rmse_clipped  = _rmse(oof_unit_clip,  y_train_unit)
val_rmse_raw      = _rmse(val_unit_raw,   y_val_unit)
val_rmse_clipped  = _rmse(val_unit_clip,  y_val_unit)
test_rmse_raw     = _rmse(test_unit_raw,  y_test_unit)
test_rmse_clipped = _rmse(test_unit_clip, y_test_unit)

print('=' * 75)
print(f'  τ_π = {TAU_PI:.4f}')
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"Raw":12s}  {oof_rmse_raw:11.6f}  {val_rmse_raw:11.6f}  {test_rmse_raw:11.6f}')
print(f'  {"Clipped":12s}  {oof_rmse_clipped:11.6f}  {val_rmse_clipped:11.6f}  {test_rmse_clipped:11.6f}')
print(f'  {"Δ":12s}  {oof_rmse_clipped-oof_rmse_raw:+11.6f}  '
      f'{val_rmse_clipped-val_rmse_raw:+11.6f}  '
      f'{test_rmse_clipped-test_rmse_raw:+11.6f}')
print('-' * 75)
print(f'  기준선:')
print(f'    bag-zit-hpo-001 #11        : val=0.005708, test=0.008411')
print(f'    bag-zit-pp-hpo-001 #4      : val=0.005709, test=0.008413')
print(f'    combined_best (no xy)      : (이 노트북 실행 후 비교)')
print('=' * 75)

killed_oof  = float((oof_die_pi  > TAU_PI).mean())
killed_val  = float((val_die_pi  > TAU_PI).mean())
killed_test = float((test_die_pi > TAU_PI).mean())
print(f'\n  τ_π 로 0 처리된 die 비율: OOF={killed_oof:.1%}, val={killed_val:.1%}, test={killed_test:.1%}')

  τ_π = 0.8689
                        OOF          val         test
  Raw              0.005509     0.005712     0.008411
  Clipped          0.005510     0.005710     0.008412
  Δ               +0.000001    -0.000001    +0.000000
---------------------------------------------------------------------------
  기준선:
    bag-zit-hpo-001 #11        : val=0.005708, test=0.008411
    bag-zit-pp-hpo-001 #4      : val=0.005709, test=0.008413
    combined_best (no xy)      : (이 노트북 실행 후 비교)

  τ_π 로 0 처리된 die 비율: OOF=13.7%, val=12.9%, test=13.1%


## 9. 아티팩트 저장

In [9]:
import json

def _build_die_df(uid_arr, die_id_arr, position_arr, pi, mu, pred, pred_clipped, y_unit):
    df = pd.DataFrame({
        KEY_COL:        uid_arr,
        DIE_KEY_COL:    die_id_arr,
        'position':     position_arr,
        'pi':           pi,
        'one_minus_pi': 1.0 - pi,
        'mu':           mu,
        'pred':         pred,
        'pred_clipped': pred_clipped,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

oof_die_df = _build_die_df(
    uid_train_die, xs_train_die[DIE_KEY_COL].values, xs_train_die['position'].values,
    oof_die_pi, oof_die_mu, oof_die_pred, oof_die_pred_clipped, y_train_unit,
)
val_die_df = _build_die_df(
    uid_val_die, xs_val_die[DIE_KEY_COL].values, xs_val_die['position'].values,
    val_die_pi, val_die_mu, val_die_pred, val_die_pred_clipped, y_val_unit,
)
test_die_df = _build_die_df(
    uid_test_die, xs_test_die[DIE_KEY_COL].values, xs_test_die['position'].values,
    test_die_pi, test_die_mu, test_die_pred, test_die_pred_clipped, y_test_unit,
)
oof_die_df.to_csv(os.path.join(OUT_DIR,  'oof_die.csv'),  index=False)
val_die_df.to_csv(os.path.join(OUT_DIR,  'val_die.csv'),  index=False)
test_die_df.to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

def _build_unit_df(unit_pred, y_unit):
    return pd.DataFrame({
        KEY_COL:  unit_pred.index.values,
        'pred':   unit_pred.values,
        'health': y_unit.reindex(unit_pred.index).values,
    })

_build_unit_df(oof_unit_raw,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(val_unit_raw,  y_val_unit  ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(test_unit_raw, y_test_unit ).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

_build_unit_df(oof_unit_clip,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit_clipped.csv'),  index=False)
_build_unit_df(val_unit_clip,  y_val_unit  ).to_csv(os.path.join(OUT_DIR, 'val_unit_clipped.csv'),  index=False)
_build_unit_df(test_unit_clip, y_test_unit ).to_csv(os.path.join(OUT_DIR, 'test_unit_clipped.csv'), index=False)

meta = {
    'exp_id':              EXP_ID,
    'model':               'BagZITboost combined val-best (HP #11 + PP #4) + position + die_x/die_y',
    'use_position':        USE_POSITION,
    'use_die_coords':      USE_DIE_COORDS,
    'tau_pi':              TAU_PI,
    'tau_pi_source':       f'avg(hpo#11={TAU_PI_FROM_HPO}, pp_hpo#4={TAU_PI_FROM_PP_HPO})',
    'n_folds':             N_FOLDS,
    'raw_oof_rmse':        oof_rmse_raw,
    'raw_val_rmse':        val_rmse_raw,
    'raw_test_rmse':       test_rmse_raw,
    'clipped_oof_rmse':    oof_rmse_clipped,
    'clipped_val_rmse':    val_rmse_clipped,
    'clipped_test_rmse':   test_rmse_clipped,
    'preprocess_PARAMS':   PARAMS,
    'effective_pp_params': pp['effective_params'],
    'fixed_hp':            {k: v for k, v in FIXED_HP.items()
                            if k not in ['random_state', 'n_jobs', 'verbose', 'device']},
    'fixed_hp_source':     'bag-zit-hpo-001 trial #11 (val-best)',
    'pp_params_source':    'bag-zit-pp-hpo-001 trial #4 (val-best)',
    'CLIP_Y_EXTREME':      CLIP_Y_EXTREME,
    'feat_cols_clean_n':   len(feat_cols_clean),
    'feat_extras':         ['position', 'die_x', 'die_y'],
    'SEED':                int(SEED),
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:35s}  {sz:10,.1f} KB')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_combined_best_xy
  meta.json                                   2.5 KB
  oof_die.csv                            13,984.0 KB
  oof_unit.csv                              971.1 KB
  oof_unit_clipped.csv                      921.4 KB
  test_die.csv                            4,664.8 KB
  test_unit.csv                             323.6 KB
  test_unit_clipped.csv                     306.7 KB
  val_die.csv                             4,664.8 KB
  val_unit.csv                              323.7 KB
  val_unit_clipped.csv                      306.8 KB


## 10. 요약

In [10]:
print('=' * 75)
print(' BagZITboost combined val-best + position + die_x/die_y — 결과 요약')
print('=' * 75)
print(f'  EXP_ID            : {EXP_ID}')
print(f'  Model HP source   : bag-zit-hpo-001 trial #11')
print(f'  PP params source  : bag-zit-pp-hpo-001 trial #4')
print(f'  position          : {USE_POSITION}')
print(f'  die_x/die_y       : {USE_DIE_COORDS}')
print(f'  τ_π               : {TAU_PI:.4f}')
print(f'  feat_cols (X+ind+pos+xy) : {len(feat_cols_clean)}')
print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"Raw":10s}  {oof_rmse_raw:11.6f}  {val_rmse_raw:11.6f}  {test_rmse_raw:11.6f}')
print(f'  {"Clipped":10s}  {oof_rmse_clipped:11.6f}  {val_rmse_clipped:11.6f}  {test_rmse_clipped:11.6f}')
print('-' * 75)
print(f'  기준선 (no xy):')
print(f'    bag-zit-hpo-001 #11        : val=0.005708, test=0.008411')
print(f'    bag-zit-pp-hpo-001 #4      : val=0.005709, test=0.008413')
print(f'    combined_best (position 만) : (해당 노트북 실행 후 비교)')
print('=' * 75)

 BagZITboost combined val-best + position + die_x/die_y — 결과 요약
  EXP_ID            : bag-zit-combined-best-xy-001
  Model HP source   : bag-zit-hpo-001 trial #11
  PP params source  : bag-zit-pp-hpo-001 trial #4
  position          : True
  die_x/die_y       : True
  τ_π               : 0.8689
  feat_cols (X+ind+pos+xy) : 571
---------------------------------------------------------------------------
                      OOF          val         test
  Raw            0.005509     0.005712     0.008411
  Clipped        0.005510     0.005710     0.008412
---------------------------------------------------------------------------
  기준선 (no xy):
    bag-zit-hpo-001 #11        : val=0.005708, test=0.008411
    bag-zit-pp-hpo-001 #4      : val=0.005709, test=0.008413
    combined_best (position 만) : (해당 노트북 실행 후 비교)
